## **GET METADATA FROM DATAHUB AND CONVERT INTO JSONL**

In [8]:
#!/usr/bin/env python3
import os
import json
import requests
import re
from typing import Dict, List, Any, Optional, Tuple
from dotenv import load_dotenv, find_dotenv
from datetime import datetime, timezone, timedelta
from urllib.parse import quote  # buat encode URN di REST

load_dotenv(find_dotenv(), override=True)

# ===== Config =====
GMS_BASE = os.getenv("DATAHUB_GMS_URL", "http://datahub-gms:8080")
GRAPHQL_URL = os.getenv("DATAHUB_GRAPHQL_URL", f"{GMS_BASE}/api/graphql")
TOKEN = os.getenv("DATAHUB_TOKEN")
if not TOKEN:
    raise RuntimeError("DATAHUB_TOKEN is not set.")
PAGE_SIZE = int(os.getenv("DATAHUB_PAGE_SIZE", "100"))
TIMEOUT = 60
ACTOR = os.getenv("DATAHUB_ACTOR")  # optional

OUTPUT_DIR = os.getenv("EXPORT_JSONL_OUTPUT", "./exports")

RELATIONAL_3LEVEL = {"postgres", "snowflake", "redshift", "bigquery", "mssql", "mysql"}

# ===== GraphQL =====
SCROLL_QUERY = """
query scrollPG($types: [EntityType!], $query: String!, $count: Int!, $scrollId: String) {
  scrollAcrossEntities(input: { types: $types, query: $query, count: $count, scrollId: $scrollId }) {
    nextScrollId
    searchResults {
      entity {
        urn
        type
        ... on Dataset { urn }
      }
    }
  }
}
""".strip()

# ✨ ADJUSTED: tambahkan editableSchemaMetadata biar dapet tag yang ditambah dari UI
DATASET_META_QUERY = """
query datasetMeta($urn: String!) {
  dataset(urn: $urn) {
    urn
    platform { name properties { datasetNameDelimiter } }
    parentContainers {
      containers {
        properties { name description }
        editableProperties { description }
      }
    }
    schemaMetadata(version: 0) {
      fields {
        fieldPath
        description
        nativeDataType
        type
        globalTags {
          tags {
            tag {
              urn
              properties { name }
            }
          }
        }
      }
    }
    editableSchemaMetadata {
      editableSchemaFieldInfo {
        fieldPath
        globalTags {
          tags {
            tag {
              urn
              properties { name }
            }
          }
        }
      }
    }
  }
}
""".strip()

DATASET_LAST_INGESTED_QUERY = """
query datasetSys($urn: String!) {
  dataset(urn: $urn) { lastIngested }
}
""".strip()

DATASET_PROFILE_QUERY = """
query datasetProfile($urn: String!) {
  dataset(urn: $urn) {
    datasetProfiles(limit: 1) {
      rowCount
      fieldProfiles {
        fieldPath
        uniqueCount
        uniqueProportion
        nullCount
        nullProportion
        min
        max
        mean
        median
        stdev
        sampleValues
      }
    }
  }
}
""".strip()


def _headers_json() -> Dict[str, str]:
    h = {"Content-Type": "application/json", "Authorization": f"Bearer {TOKEN}"}
    if ACTOR:
        h["X-DataHub-Actor"] = ACTOR
    return h


def _graphql(query: str, variables: dict) -> dict:
    resp = requests.post(
        GRAPHQL_URL,
        json={"query": query, "variables": variables},
        headers=_headers_json(),
        timeout=TIMEOUT,
    )
    resp.raise_for_status()
    data = resp.json()
    if data.get("errors"):
        raise RuntimeError(data["errors"][0])
    return data["data"]


def iter_dataset_urns():
    scroll_id = None
    while True:
        data = _graphql(
            SCROLL_QUERY,
            {
                "types": ["DATASET"],
                "query": "*",
                "count": PAGE_SIZE,
                "scrollId": scroll_id,
            },
        )["scrollAcrossEntities"]
        for item in data.get("searchResults") or []:
            ent = item.get("entity") or {}
            urn = ent.get("urn")
            if urn:
                yield urn
        scroll_id = data.get("nextScrollId")
        if not scroll_id:
            break


def _parse_urn_name(urn: str) -> Optional[str]:
    try:
        inner = urn[urn.index("(") + 1 : urn.rindex(")")]
        _, name_part, _ = inner.split(",", 2)
        return name_part
    except Exception:
        return None


def _split_from_urn(
    urn: str, platform_name: Optional[str], delim: str = "."
) -> Tuple[Optional[str], Optional[str], Optional[str], Optional[str]]:
    name_part = _parse_urn_name(urn)
    if not name_part:
        return None, None, None, None
    plat = (platform_name or "").lower()
    if plat in RELATIONAL_3LEVEL:
        parts = name_part.split(delim)
        if len(parts) >= 3:
            return name_part, parts[0], parts[1], delim.join(parts[2:])
        elif len(parts) == 2:
            return name_part, parts[0], None, parts[1]
        else:
            return name_part, None, None, parts[0]
    return name_part, None, None, name_part


def _best_effort_last_ingested(urn: str) -> Optional[int]:
    try:
        d = _graphql(DATASET_LAST_INGESTED_QUERY, {"urn": urn})
        ds = d.get("dataset")
        if ds is not None:
            return ds.get("lastIngested")
    except Exception:
        pass
    return None


# ---- time helpers ----
_JAKARTA = timezone(timedelta(hours=7))


def _ms_to_local_naive(ms: Optional[int]) -> Optional[str]:
    if ms is None:
        return None
    dt_local = datetime.fromtimestamp(ms / 1000, tz=timezone.utc).astimezone(_JAKARTA)
    return dt_local.strftime("%Y-%m-%dT%H:%M:%S")


# ---- profiling helpers ----
_NUMERIC_KEYWORDS = (
    "int",
    "integer",
    "smallint",
    "bigint",
    "tinyint",
    "double",
    "float",
    "real",
    "decimal",
    "numeric",
    "number",
    "money",
)


def _is_numeric(datatype: Optional[str]) -> bool:
    if not datatype:
        return False
    s = str(datatype).lower()
    return any(k in s for k in _NUMERIC_KEYWORDS)


def _norm_field(s: Optional[str]) -> Optional[str]:
    if s is None:
        return None
    return s.strip().strip('`"[]').lower()


def _as_float(x):
    if x is None:
        return None
    try:
        return float(x)
    except Exception:
        return None


def _pick_sample_value(fp: dict) -> str:
    sv = fp.get("sampleValues") or []
    if isinstance(sv, list) and sv:
        return str(sv[0])
    return ""


def _profile_map_by_field(urn: str) -> dict:
    try:
        d = _graphql(DATASET_PROFILE_QUERY, {"urn": urn})
    except Exception as e:
        print(f"[profile] GraphQL error for {urn}: {e}")
        return {}

    ds = (d.get("dataset") or {})
    prof = (ds.get("datasetProfiles") or [None])[0]
    if not prof:
        return {}

    out = {}
    for fp in (prof.get("fieldProfiles") or []):
        key = _norm_field(fp.get("fieldPath"))
        if not key:
            continue

        min_v = _as_float(fp.get("min"))
        max_v = _as_float(fp.get("max"))
        mean_v = _as_float(fp.get("mean"))
        stdev_v = _as_float(fp.get("stdev"))

        out[key] = {
            "nullCount": fp.get("nullCount"),
            "min": min_v,
            "max": max_v,
            "mean": mean_v,
            "stddev": stdev_v,
            "distinct_count": fp.get("uniqueCount"),
            "sample_value": _pick_sample_value(fp),
        }
    return out


# ===== REST FALLBACK FOR TAGS =====
def fetch_schema_via_rest(urn: str) -> Optional[dict]:
    """
    REST fallback supaya bisa baca schemaMetadata yang berisi field.globalTags.
    """
    url = f"{GMS_BASE}/entities/{quote(urn, safe='')}"
    params = {"aspect": "schemaMetadata", "version": "0"}
    headers = {"Authorization": f"Bearer {TOKEN}"}
    if ACTOR:
        headers["X-DataHub-Actor"] = ACTOR
    try:
        resp = requests.get(url, headers=headers, params=params, timeout=TIMEOUT)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        print(f"[rest] cannot fetch schemaMetadata via REST for {urn}: {e}")
        return None

    entity_resp = data.get("entityResponse") or data
    aspects = entity_resp.get("aspects") or {}
    sm = aspects.get("schemaMetadata")
    if not sm:
        return None

    if isinstance(sm, dict) and "value" in sm:
        return sm["value"]
    return sm


def build_field_tags_map_from_rest_schema(rest_schema: dict) -> Dict[str, List[str]]:
    out: Dict[str, List[str]] = {}
    if not rest_schema:
        return out
    for f in rest_schema.get("fields") or []:
        fp = f.get("fieldPath")
        if not fp:
            continue
        gtags = ((f.get("globalTags") or {}).get("tags") or [])
        names: List[str] = []
        for gt in gtags:
            tag_val = gt.get("tag")
            if isinstance(tag_val, str):
                names.append(tag_val)
            elif isinstance(tag_val, dict):
                name = (tag_val.get("properties") or {}).get("name") or tag_val.get("urn")
                if name:
                    names.append(name)
        out[fp] = names
    return out


def split_alias_and_role(raw_tags: List[str]) -> Tuple[List[str], List[str]]:
    """
    Pisahin tag jadi 2 list:
    - alias -> buang prefix alias: / alias.
    - role  -> buang prefix role: / role.
    """
    alias_out: List[str] = []
    role_out: List[str] = []
    for t in raw_tags:
        # kalau bentuknya urn:li:tag:alias.foo -> ambil yang belakangnya dulu
        if t.startswith("urn:li:tag:"):
            short = t[len("urn:li:tag:") :]
        else:
            short = t

        # normalize ke lower biar gampang
        lowered = short.lower()

        if lowered.startswith("alias:") or lowered.startswith("alias."):
            # potong prefix 6 char ("alias:") atau ganti titik jadi kolon dulu
            cleaned = re.sub(r"^alias[:.]", "", short, flags=re.IGNORECASE).strip()
            if cleaned:
                alias_out.append(cleaned)
        elif lowered.startswith("role:") or lowered.startswith("role."):
            cleaned = re.sub(r"^role[:.]", "", short, flags=re.IGNORECASE).strip()
            if cleaned:
                role_out.append(cleaned)
        else:
            # tag lain cuekin / atau bisa kamu masukin ke alias_out kalau mau
            pass

    return alias_out, role_out



# ===== MAIN FETCH =====
def fetch_dataset_columns(urn: str) -> List[Dict[str, Any]]:
    d = _graphql(DATASET_META_QUERY, {"urn": urn}).get("dataset")
    if not d:
        return []

    platform = d.get("platform") or {}
    platform_name = (platform.get("name") or "").lower()
    delim = (platform.get("properties") or {}).get("datasetNameDelimiter") or "."

    _, db, schema, table = _split_from_urn(urn, platform_name, delim)

    # parent container desc
    db_desc = ""
    containers = ((d.get("parentContainers") or {}).get("containers") or [])
    for c in containers:
        props = c.get("properties") or {}
        edit = c.get("editableProperties") or {}
        if (props.get("name") or "").lower() == (db or "").lower():
            db_desc = (edit.get("description") or props.get("description") or "").strip()
            break
    if not db_desc:
        for c in containers:
            props = c.get("properties") or {}
            edit = c.get("editableProperties") or {}
            cand = (edit.get("description") or props.get("description") or "").strip()
            if cand:
                db_desc = cand
                break

    last_ingested_ms = _best_effort_last_ingested(urn)
    last_ingested_str = _ms_to_local_naive(last_ingested_ms)

    profile_map = _profile_map_by_field(urn)
    omit_schema = platform_name == "mysql"

    # --- 1) REST fallback tag map
    rest_schema = fetch_schema_via_rest(urn)
    rest_field_tags_map = build_field_tags_map_from_rest_schema(rest_schema) if rest_schema else {}

    # --- 2) editable field tag map from GraphQL
    editable = (d.get("editableSchemaMetadata") or {}).get("editableSchemaFieldInfo") or []
    editable_tag_map: Dict[str, List[str]] = {}
    for e in editable:
        fp = e.get("fieldPath")
        if not fp:
            continue
        gtags = ((e.get("globalTags") or {}).get("tags") or [])
        names: List[str] = []
        for gt in gtags:
            tag = gt.get("tag") or {}
            # kalau GraphQL, biasanya {urn, properties{name}}
            name = (tag.get("properties") or {}).get("name") or tag.get("urn")
            if name:
                names.append(name)
        editable_tag_map[fp] = names

    rows: List[Dict[str, Any]] = []
    sm = d.get("schemaMetadata") or {}
    for f in (sm.get("fields") or []):
        field_path = f.get("fieldPath")
        native = f.get("nativeDataType") or (f.get("type") or "")
        is_num = _is_numeric(native)

        prof = profile_map.get(_norm_field(field_path), {}) if profile_map else {}
        sample_out = prof.get("sample_value", "")

        if is_num:
            min_v = prof.get("min")
            max_v = prof.get("max")
            mean_v = prof.get("mean")
            std_v = prof.get("stddev")
        else:
            min_v = max_v = mean_v = std_v = "unknown"

        # --- 3 sumber tag:
        # a) dari schemaMetadata (kalau ada)
        schema_tags_raw: List[str] = []
        gtags_schema = ((f.get("globalTags") or {}).get("tags") or [])
        for gt in gtags_schema:
            tag = gt.get("tag") or {}
            name = (tag.get("properties") or {}).get("name") or tag.get("urn")
            if name:
                schema_tags_raw.append(name)

        # b) dari editableSchemaMetadata
        editable_raw = editable_tag_map.get(field_path, [])

        # c) dari REST fallback
        rest_raw = rest_field_tags_map.get(field_path, [])

        # gabungkan semuanya dan dedup urutannya
        merged_raw = list(dict.fromkeys(schema_tags_raw + editable_raw + rest_raw))

        tags_alias, tags_role = split_alias_and_role(merged_raw)

        row: Dict[str, Any] = {
            "urn": urn,
            "field_urn": f"{urn}::{field_path}",
            "database": db,
            "table": table,
            "column": field_path,
            "datatype": native,
            "description": f.get("description") or "",
            "database_description": db_desc,
            "last_ingested": last_ingested_str,
            "min": min_v,
            "max": max_v,
            "mean": mean_v,
            "stddev": std_v,
            "sample_value": sample_out,
            "tags_alias": tags_alias,
            "tags_role": tags_role,
        }
        if (schema is not None) and (not omit_schema):
            row["schema"] = schema
        if prof.get("distinct_count") is not None:
            row["distinct_count"] = prof["distinct_count"]

        rows.append(row)

    return rows


def _sanitize_filename(name: str) -> str:
    if not name:
        return "unknown"
    s = name.strip().lower()
    s = re.sub(r"[^a-z0-9\\-_.]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s or "unknown"


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    writers: Dict[str, Any] = {}
    counts: Dict[str, int] = {}

    def get_writer(db_name: Optional[str]):
        db_key = db_name or "unknown"
        safe = _sanitize_filename(db_key)
        path = os.path.join(OUTPUT_DIR, f"{safe}.jsonl")
        if safe not in writers:
            writers[safe] = open(path, "w", encoding="utf-8")
            counts[safe] = 0
        return writers[safe], safe

    try:
        for urn in iter_dataset_urns():
            try:
                rows = fetch_dataset_columns(urn)
                print(f"[export] {urn} → {len(rows)} columns")
                for row in rows:
                    writer, key = get_writer(row.get("database"))
                    writer.write(json.dumps(row, ensure_ascii=False) + "\n")
                    counts[key] += 1
            except Exception as e:
                writer, key = get_writer("unknown")
                writer.write(
                    json.dumps({"error": f"{urn}: {e}"}, ensure_ascii=False) + "\n"
                )
                counts[key] += 1
    finally:
        for fh in writers.values():
            try:
                fh.close()
            except Exception:
                pass

    print(f"Exported per database to: {os.path.abspath(OUTPUT_DIR)}")
    for db_file, n in sorted(counts.items()):
        print(f"- {db_file}.jsonl: {n} rows")


if __name__ == "__main__":
    main()

[export] urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.customers,PROD) → 13 columns
[export] urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.employees,PROD) → 8 columns
[export] urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.offices,PROD) → 9 columns
[export] urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.orderdetails,PROD) → 5 columns
[export] urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.orders,PROD) → 7 columns
[export] urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.payments,PROD) → 4 columns
[export] urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.productlines,PROD) → 4 columns
[export] urn:li:dataset:(urn:li:dataPlatform:mysql,classicmodels.products,PROD) → 9 columns
Exported per database to: /app/metadata_jsonl
- classicmodels.jsonl: 59 rows
